# Extract the EL CVn's   


### Discovery of 36 eclipsing EL CVn binaries found by the Palomar Transient Factory 
### Van Roestel et al. 2018


https://ui.adsabs.harvard.edu/abs/2018MNRAS.475.2560V/abstract

**They list the systems with their parameters in Appendix table A (raw latex shown below)**





## First import the latex table into a df

In [ ]:
# Lets start with some imports
import pandas as pd
import numpy as np
import re
import json
import sys
from pathlib import Path
from textwrap import dedent

proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

# -------------------------------------------------
# Configuration
# -------------------------------------------------
from paths import RESULT_TABLES, RAW_JSON_DIR, DATA_DIR

In [ ]:


latex = dedent(r"""
<put the LaTeX table text here, from \begin{tabular} through the end>
""")

# Extract only data lines (skip \hline, captions, etc.)
lines = [
    ln for ln in latex.splitlines()
    if '&' in ln and r'\hline' not in ln and r'\contcaption' not in ln
]

def parse_line(line):
    parts = [p.strip().strip('$ ') for p in line.split('&')]
    name_raw = parts[0].replace(r'^\mathrm{RV}', '^RV')
    name = re.sub(r'\\\^\{\s*\\mathrm\{RV\}\s*\}', '^RV', name_raw)
    name = re.sub(r'[{}\\$]', '', name).strip()
    nums = [float(re.sub(r'[^\d\.\-eE]', '', p)) for p in parts[1:]]
    return name, nums

records = []
for val_line, err_line in zip(lines[0::2], lines[1::2]):  # pair value/err rows
    name, vals = parse_line(val_line)
    _, errs = parse_line(err_line)
    records.append((name, vals, errs))

cols = [
    "P_d", "i_deg", "M1_Msun", "M2_Msun",
    "R1_Rsun", "R2_Rsun", "T1_K", "T2_K",
    "logg1", "logg2"
]
df = pd.DataFrame([
    {"name": name, **{c: v for c, v in zip(cols, vals)},
     **{f"{c}_err": e for c, e in zip(cols, errs)}}
    for name, vals, errs in records
])

df.head()

""


In [ ]:
# Set 

Columns: ['main_id', 'ra', 'dec', 'coo_err_maj', 'coo_err_min', 'coo_err_angle', 'coo_wavelength', 'coo_bibcode', 'matched_id']
Row 0:     main_id             ra              dec       coo_err_maj coo_err_min coo_err_angle coo_wavelength     coo_bibcode      matched_id 
                       deg              deg           mas         mas          deg                                                     
--------------- ------------------ -------------- ----------- ----------- ------------- -------------- ------------------- ------------
TYC 5204-1575-1 315.37195674043994 -6.37080081665       0.029      0.0171            90              O 2020yCat.1350....0G TIC 35399970


# Draw the RA, DEC and some spectral type info from Simbad

In [15]:
# Query SIMBAD for RA/Dec coordinates (and spectral type if available)
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u

# Configure Simbad to return decimal degrees and spectral type
simbad = Simbad()
simbad.TIMEOUT = 30
simbad.add_votable_fields('ra(d)', 'dec(d)', 'ra', 'dec', 'sp')

ra_list = []
dec_list = []
sp_list = []

for tic in df['TIC']:
    try:
        result = simbad.query_object(f'TIC {tic}')
        if result is None or len(result) == 0:
            ra_list.append(None)
            dec_list.append(None)
            sp_list.append(None)
            print(f"TIC {tic}: Not found in SIMBAD")
            continue

        ra_deg = None
        dec_deg = None
        sp_val = None
        cols = result.colnames

        if 'ra' in cols and 'dec' in cols:
            ra_deg = float(result[0]['ra'])
            dec_deg = float(result[0]['dec'])
        else:
            coord = SkyCoord(result[0]['RA'], result[0]['DEC'], unit=(u.hourangle, u.deg))
            ra_deg = coord.ra.deg
            dec_deg = coord.dec.deg

        if 'SP_TYPE' in cols:
            sp_val = result[0]['SP_TYPE']
        elif 'sp_type' in cols:
            sp_val = result[0]['sp_type']
        elif 'SP' in cols:
            sp_val = result[0]['SP']

        if sp_val is not None and hasattr(sp_val, 'mask') and getattr(sp_val, 'mask', False):
            sp_val = None
        if sp_val is not None:
            sp_val = str(sp_val).strip() or None

        ra_list.append(ra_deg)
        dec_list.append(dec_deg)
        sp_list.append(sp_val)
        if ra_deg is not None and dec_deg is not None:
            if sp_val:
                print(f"TIC {tic}: RA={ra_deg:.6f} deg, Dec={dec_deg:.6f} deg, SpType={sp_val}")
            else:
                print(f"TIC {tic}: RA={ra_deg:.6f} deg, Dec={dec_deg:.6f} deg")
        else:
            print(f"TIC {tic}: Coordinates missing despite SIMBAD hit (columns: {cols})")
    except Exception as e:
        ra_list.append(None)
        dec_list.append(None)
        sp_list.append(None)
        print(f"TIC {tic}: Query error - {e}")

# Add RA/Dec/SpType to dataframe
df['RA'] = ra_list
df['Dec'] = dec_list
df['SpType'] = sp_list

print(f"\nAdded coordinates for {sum(1 for x in ra_list if x is not None)} systems")
print(f"Spectral types found for {sum(1 for x in sp_list if x)} systems")
display(df[['TIC', 'RA', 'Dec', 'SpType', 'Period', 'M1', 'M2']])

/var/folders/5d/vcxrsh5975l7d5n8t7vvc5pr0000gn/T/ipykernel_60791/3547983570.py:9: DeprecationWarning: 'ra(d)' has been renamed 'ra'. You'll see it appearing with its new name in the output table
  simbad.add_votable_fields('ra(d)', 'dec(d)', 'ra', 'dec', 'sp')
/var/folders/5d/vcxrsh5975l7d5n8t7vvc5pr0000gn/T/ipykernel_60791/3547983570.py:9: DeprecationWarning: 'dec(d)' has been renamed 'dec'. You'll see it appearing with its new name in the output table
  simbad.add_votable_fields('ra(d)', 'dec(d)', 'ra', 'dec', 'sp')


TIC 35399970: RA=315.371957 deg, Dec=-6.370801 deg
TIC 83833793: RA=217.465020 deg, Dec=-24.724273 deg, SpType=F0
TIC 121078334: RA=56.598694 deg, Dec=-21.972104 deg, SpType=A4IV
TIC 160081043: RA=352.053172 deg, Dec=-39.923251 deg
TIC 35399970: RA=315.371957 deg, Dec=-6.370801 deg
TIC 166874908: RA=59.659724 deg, Dec=-31.277309 deg, SpType=A3
TIC 192990023: RA=155.345141 deg, Dec=-28.694330 deg
TIC 408351887: RA=144.990610 deg, Dec=-19.328100 deg
TIC 149160359: RA=81.455091 deg, Dec=-64.989374 deg, SpType=A7V
TIC 416264037: RA=289.608635 deg, Dec=48.884104 deg, SpType=kA4hA7mA9
TIC 100011519: RA=173.003028 deg, Dec=60.112337 deg
TIC 219485855: RA=242.846723 deg, Dec=44.105816 deg, SpType=A3
TIC 399725538: RA=73.269408 deg, Dec=10.252742 deg, SpType=kA2hA7mA6
TIC 464641792: RA=287.656617 deg, Dec=-61.125651 deg
TIC 54957535: RA=130.985253 deg, Dec=-11.557658 deg
TIC 142258314: RA=131.495033 deg, Dec=53.036040 deg
TIC 197604137: RA=336.755745 deg, Dec=48.832474 deg, SpType=A5
TIC 400028

,TIC,RA,Dec,SpType,Period,M1,M2
0,35399970,315.371957,-6.370801,None,"[nan, 1.2908616, nan]","[0.02, 1.85, 0.02]","[0.002, 0.193, 0.002]"
1,83833793,217.465020,-24.724273,F0,"[nan, 2.17351686, nan]","[0.02, 2.05, 0.02]","[0.03, 0.2, 0.03]"
2,121078334,56.598694,-21.972104,A4IV,"[nan, 0.92859504, nan]","[0.01, 1.68, 0.02]","[0.002, 0.185, 0.002]"
3,160081043,352.053172,-39.923251,None,"[nan, 0.768701771, nan]","[0.01, 1.69, 0.01]","[0.001, 0.196, 0.001]"
4,35399970,315.371957,-6.370801,None,"[nan, 1.2908616, nan]","[0.02, 1.85, 0.02]","[0.002, 0.193, 0.002]"
5,166874908,59.659724,-31.277309,A3,"[nan, 2.18931, nan]","[0.01, 1.95, 0.01]","[0.004, 0.197, 0.004]"
6,192990023,155.345141,-28.694330,None,"[nan, 0.900898203, nan]","[0.04, 1.53, 0.04]","[0.005, 0.193, 0.005]"
7,408351887,144.990610,-19.328100,None,"[nan, 1.0731802, nan]","[0.01, 1.74, 0.01]","[0.005, 0.196, 0.005]"
8,149160359,81.455091,-64.989374,A7V,"[nan, 1.120738, nan]","[0.001, 1.88, 0.001]","[0.001, 0.188, 0.001]"
9,416264037,289.608635,48.884104,kA4hA7mA9,"[nan, 1.15991, nan]","[0.001, 1.66, 0.001]","[0.001, 0.178, 0.001]"


In [ ]:
# https://simbad.u-strasbg.fr/simbad/sim-ref?querymethod=bib&simbo=on&submit=submit+bibcode&bibcode=2025ApJ...979..108X




SyntaxError: invalid decimal literal (3749623238.py, line 1)

# Now pour this into a json table

In [17]:
# Convert DataFrame to schema format
systems = []
for idx, row in df.iterrows():
    ra_triplet = [None, row['RA'], None] if row['RA'] is not None else [None, None, None]
    dec_triplet = [None, row['Dec'], None] if row['Dec'] is not None else [None, None, None]
    sp_type = row['SpType'] if 'SpType' in row and pd.notna(row['SpType']) and row['SpType'] else None

    system = {
        'System Name': f'TIC {row["TIC"]}',
        'RA': ra_triplet,
        'Dec': dec_triplet,
        'Period': row['Period'],
        'Eccentricity': [None, None, None],
        'M1': row['M1'],
        'M2': row['M2'],
        'Mass Function': [None, None, None],
        'evol_type_1': 'MS',
        'evol_type_2': 'WD',
        'obs_type_1': sp_type,
        'obs_type_2': 'pre-He WD',
        'system_class': 'EL CVn',
        'Detection Method': ['EB'],
        'Reference': ['2025ApJ...979..108X'],
        'Notes': f'Absolute parameters for EL CVn-type binary from Xiong et al. 2025. Source: {["previously discovered with parameters", "previously discovered", "newly discovered"][row["Source"]-1]}. Additional measurements: T1={row["T1"]}, T2={row["T2"]}, logg1={row["logg1"]}, logg2={row["logg2"]}, R1={row["R1"]}, R2={row["R2"]}',
        'Simbad': f'https://simbad.cds.unistra.fr/simbad/sim-id?Ident=TIC+{int(row["TIC"])}' if row['RA'] is not None else None
    }
    systems.append(system)

output_file = Path(RESULT_TABLES) / 'raw_json' / 'Xiong2025_ELCVns.raw.json'
with open(output_file, "w") as f:
    f.write("[\n")
    for i, system in enumerate(systems):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        f.write("  " + line)
        if i < len(systems) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")

print(f"Saved {len(systems)} EL CVn systems to {output_file}")
print("\nFirst system example:")
print(json.dumps(systems[0], indent=2))

Saved 31 EL CVn systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Xiong2025_ELCVns.raw.json

First system example:
{
  "System Name": "TIC 35399970",
  "RA": [
    null,
    315.37195674043994,
    null
  ],
  "Dec": [
    null,
    -6.37080081665,
    null
  ],
  "Period": [
    NaN,
    1.2908616,
    NaN
  ],
  "Eccentricity": [
    null,
    null,
    null
  ],
  "M1": [
    0.02,
    1.85,
    0.02
  ],
  "M2": [
    0.002,
    0.193,
    0.002
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "evol_type_1": "MS",
  "evol_type_2": "WD",
  "obs_type_1": null,
  "obs_type_2": "pre-He WD",
  "system_class": "EL CVn",
  "Detection Method": [
    "EB"
  ],
  "Reference": [
    "2025ApJ...979..108X"
  ],
  "Notes": "Absolute parameters for EL CVn-type binary from Xiong et al. 2025. Source: previously discovered with parameters. Additional measurements: T1=[39.0, 8597.0, 68.0], T2=[82.0, 11378.0, 83.0], logg1=[0.003, 4.294, 0.003], lo